# Finetuning Models for upload to Ollama

In [1]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Failed to import trl openenv: No module named 'trl.experimental'


==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 4. Max memory: 79.209 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.12.4 patched 80 layers with 80 QKV layers, 80 O layers and 80 MLP layers.


In [4]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.3",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

In [5]:
import pandas as pd
import json

correct_edges_df = pd.read_json('./data/p2_responses.json')
raw_dataset = correct_edges_df.to_dict(orient='records')
print(raw_dataset)
len(raw_dataset)
for i in range(len(raw_dataset)):
    convos = raw_dataset[i]
    for j in range(len(convos['conversations'])):
        convo = convos['conversations'][j]
        for key, value in convo.items():
            if not isinstance(value, str):
                raw_dataset[i]['conversations'][j][key] = str(value)

print(raw_dataset)

[{'conversations': [{'from': 'input', 'value': '\n        Act as a data analyzer for cybersecurity graph relationships.\n\n        Given these two nodes:\n        {\n  "src_node": {\n    "_key": "Planning_with_Content_Dev_Team",\n    "_id": "PlanningStep/Planning_with_Content_Dev_Team",\n    "_rev": "_kl29R5K---",\n    "name": "Planning with Content Dev Team",\n    "description": "Planning with Content Dev Team.",\n    "is_initial_step": true,\n    "next_steps": [\n      "DevelopmentStep/Draft_campaign_plan_with_Content_Dev_Team"\n    ],\n    "artifacts": [\n      "CustomerRequirementArtifact/obap_cust_req_001",\n      "CapabilityRequestArtifact/obap_cap_req_001",\n      "StorylineArtifact/obap_storyline_001"\n    ]\n  },\n  "pair_node": {\n    "_key": "Draft_campaign_plan_with_Content_Dev_Team",\n    "_id": "DevelopmentStep/Draft_campaign_plan_with_Content_Dev_Team",\n    "_rev": "_kl29R56---",\n    "name": "Draft campaign plan with Content Dev Team",\n    "description": "Draft campai

In [6]:
from datasets import Dataset
dataset = Dataset.from_list(raw_dataset)

In [7]:
dataset[0]

{'conversations': [{'from': 'input',
   'value': '\n        Act as a data analyzer for cybersecurity graph relationships.\n\n        Given these two nodes:\n        {\n  "src_node": {\n    "_key": "Planning_with_Content_Dev_Team",\n    "_id": "PlanningStep/Planning_with_Content_Dev_Team",\n    "_rev": "_kl29R5K---",\n    "name": "Planning with Content Dev Team",\n    "description": "Planning with Content Dev Team.",\n    "is_initial_step": true,\n    "next_steps": [\n      "DevelopmentStep/Draft_campaign_plan_with_Content_Dev_Team"\n    ],\n    "artifacts": [\n      "CustomerRequirementArtifact/obap_cust_req_001",\n      "CapabilityRequestArtifact/obap_cap_req_001",\n      "StorylineArtifact/obap_storyline_001"\n    ]\n  },\n  "pair_node": {\n    "_key": "Draft_campaign_plan_with_Content_Dev_Team",\n    "_id": "DevelopmentStep/Draft_campaign_plan_with_Content_Dev_Team",\n    "_rev": "_kl29R56---",\n    "name": "Draft campaign plan with Content Dev Team",\n    "description": "Draft camp

In [8]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset, aliases_for_assistant=['CAL', 'gpt', 'assistant', 'output'])
dataset = dataset.map(formatting_prompts_func, batched = True,)

num_proc must be <= 71. Reducing num_proc to 71 for dataset of size 71.
[datasets.arrow_dataset|WARNING]num_proc must be <= 71. Reducing num_proc to 71 for dataset of size 71.


Unsloth: Standardizing formats (num_proc=71):   0%|          | 0/71 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [9]:
dataset[0]

{'conversations': [{'content': '\n        Act as a data analyzer for cybersecurity graph relationships.\n\n        Given these two nodes:\n        {\n  "src_node": {\n    "_key": "Planning_with_Content_Dev_Team",\n    "_id": "PlanningStep/Planning_with_Content_Dev_Team",\n    "_rev": "_kl29R5K---",\n    "name": "Planning with Content Dev Team",\n    "description": "Planning with Content Dev Team.",\n    "is_initial_step": true,\n    "next_steps": [\n      "DevelopmentStep/Draft_campaign_plan_with_Content_Dev_Team"\n    ],\n    "artifacts": [\n      "CustomerRequirementArtifact/obap_cust_req_001",\n      "CapabilityRequestArtifact/obap_cap_req_001",\n      "StorylineArtifact/obap_storyline_001"\n    ]\n  },\n  "pair_node": {\n    "_key": "Draft_campaign_plan_with_Content_Dev_Team",\n    "_id": "DevelopmentStep/Draft_campaign_plan_with_Content_Dev_Team",\n    "_rev": "_kl29R56---",\n    "name": "Draft campaign plan with Content Dev Team",\n    "description": "Draft campaign plan with Con

In [10]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/71 [00:00<?, ? examples/s]

In [11]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=64):   0%|          | 0/71 [00:00<?, ? examples/s]

In [12]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n\n        Act as a data analyzer for cybersecurity graph relationships.\n\n        Given these two nodes:\n        {\n  "src_node": {\n    "_key": "Requirement_Gathering_with_OPFOR_and_Automation",\n    "_id": "PlanningStep/Requirement_Gathering_with_OPFOR_and_Automation",\n    "_rev": "_kl29R86---",\n    "name": "Requirement Gathering with OPFOR and Automation",\n    "description": "Requirement gathering with OPFOR and Automation Team.",\n    "is_initial_step": true,\n    "next_steps": [\n      "DevelopmentStep/Capability_Building_with_OPFOR_Automation"\n    ],\n    "artifacts": [\n      "RangeRequestArtifact/obap_range_req_001"\n    ]\n  },\n  "pair_node": {\n    "_key": "Capability_Building_with_OPFOR_Automation",\n    "_id": "DevelopmentStep/Capability_Building_with_OPFOR_Automation"

In [13]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])


'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         {\'src_attr\': \'next_steps\', \'dest_attr\': \'_id\', \'type\': \'LEADS_TO\', \'explanation\': "The src_node\'s next_steps field contains the pair_node\'s _id, indicating a sequential relationship where the source directly precedes the destination in time/workflow"}<|eot_id|>'

In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 71 | Num Epochs = 7 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 207,093,760 of 70,760,800,256 (0.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.540300
2,0.504900
3,0.443500
4,0.427600
5,0.104600
6,0.116800
7,0.190500
8,0.133100
9,0.150500
10,0.113700


In [15]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA H100 80GB HBM3. Max memory = 79.209 GB.
42.572 GB of memory reserved.


In [16]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")


398.3651 seconds used for training.
6.64 minutes used for training.
Peak reserved memory = 42.572 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 53.746 %.
Peak reserved memory for training % of max memory = 0.0 %.


In [17]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContinue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe next numbers in the Fibonacci sequence are:\n\n1, 1, 2, 3, 5, 8, 13, 21, 34, 55,...\n\nEach number in the sequence is the sum of the two preceding numbers, starting from 1 and 1.<|eot_id|>']

In [20]:
model.save_pretrained_gguf("llama3-3-70b-tuned-it-f16", tokenizer, quantization_method='f16')

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/admin/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00030.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/30 [00:00<?, ?it/s]

model-00001-of-00030.safetensors:   0%|          | 0.00/4.58G [00:00<?, ?B/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 30/30 [18:25<00:00, 36.84s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 30/30 [04:57<00:00,  9.92s/it]


Unsloth: Merge process complete. Saved to `/home/admin/unsloth_venv/llama3-3-70b-tuned-it-f16`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to convert model - output file llama-3.3-70b-instruct.BF16.gguf not created

In [ ]:
!cat ./llama-3.3-70b-instruct.BF16-00001-of-00003.gguf ./llama-3.3-70b-instruct.BF16-00002-of-00003.gguf ./llama-3.3-70b-instruct.BF16-00003-of-00003.gguf > ./llama3-3-70b-tuned-it-f16/llama-3.3-70b-instruct.BF16.gguf

In [ ]:
modelfile_path = './llama3-3-70b-tuned-it-f16/Modelfile'

modelfile_contents = """
FROM ./llama3-3-70b-tuned-it-f16/llama3-3-70b-instruct.BF16.gguf

PARAMETER temperature 0.9

SYSTEM You are a data assistant assessing knowledge graph connections.
"""

with open(modelfile_path, 'w') as file:
    file.write(modelfile_contents)

In [ ]:
!cat ./llama3-3-70b-tuned-it-f16/Modelfile
!ollama create llama3.3:70b-it-q4km-cust-v1 -q q4_K_M -f ./llama3-3-70b-tuned-it-f16/llama-3.3-70b.Modelfile


FROM ./llama3-3-70b-tuned-it-q4km-cust-v1/llama3-3-70b-instruct.BF16.gguf

PARAMETER temperature 0.9

SYSTEM You are a data assistant assessing knowledge graph connections.
gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠋ gathering model components ⠙ gathering model components ⠙ gathering model components ⠸ gathering model components ⠸ gathering model components ⠴ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠦ gathering model components ⠧ gathering mode

In [16]:
model.save_pretrained_merged("llama3-3-70b-tuned-it-f16", tokenizer,save_method='merged_16bit')

Found HuggingFace hub cache directory: /home/admin/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00030.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 30/30 [00:00<00:00, 91846.07it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 30/30 [04:29<00:00,  8.98s/it]


Unsloth: Merge process complete. Saved to `/home/admin/unsloth_venv/llama3-3-70b-tuned-it-f16`


In [29]:
model.save_pretrained("llama3-3-70b-tuned-it-q4km")  # Local saving
tokenizer.save_pretrained("llama3-3-70b-tuned-it-q4km")

('llama3-3-70b-tuned-it-q4km/tokenizer_config.json',
 'llama3-3-70b-tuned-it-q4km/special_tokens_map.json',
 'llama3-3-70b-tuned-it-q4km/chat_template.jinja',
 'llama3-3-70b-tuned-it-q4km/tokenizer.json')

In [16]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "./llama3-3-70b-tuned-it-q4km", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [                    # Change below!
    raw_dataset[0]['conversations'][0]
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)


==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 4. Max memory: 79.209 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

UndefinedError: 'dict object' has no attribute 'role'